In [ ]:
import sys
sys.path.append('..')
import numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation, metric_fit, mesh, rigid_registration
from numpy.linalg import norm

In [ ]:
m3d = mesh.Mesh('../../examples/lilium.msh')
F = m3d.triangles()
uv = np.loadtxt('data/lilium_param.txt')
phi = np.loadtxt('data/lilium_contraction_angle.txt')
invSigmas = np.loadtxt('data/lilium_singular_values.txt')

u0 = np.column_stack((np.cos(phi), np.sin(phi)))
u1 = np.column_stack((np.cos(phi + np.pi / 2), np.sin(phi + np.pi / 2)))

sigma0 = 1.0 / invSigmas[:, 0]
sigma1 = 1.0 / invSigmas[:, 1]

In [ ]:
sigma1.fill(1.0)

In [ ]:
g = (np.einsum('ij,ik->ijk', sigma0[:, None]**2 * u0, u0) +
     np.einsum('ij,ik->ijk', sigma1[:, None]**2 * u1, u1))

In [ ]:
mflat = metric_fit.Mesh2D(uv, F)
fitter = metric_fit.MetricFitter(mflat)
fitter.setTargetMetric(g)

In [ ]:
VRegistered = m3d.vertices() - np.array([0, 0, np.mean(m3d.vertices()[:, 2])])

In [ ]:
uv3d = np.pad(uv, [(0, 0), (0, 1)], mode='constant')
R, t = rigid_registration.register(uv3d, m3d.vertices())
uv3d = uv3d @ R.transpose() + t

In [ ]:
uv3d[m3d.boundaryVertices(), :] = m3d.vertices()[m3d.boundaryVertices()]
fitter.setVars(uv3d.ravel())

In [ ]:
P = m3d.vertices()[m3d.boundaryVertices()]

In [ ]:
t = -np.mean(P, axis=0)
Pshift = P + t
R = np.linalg.eig(Pshift.transpose() @ Pshift)[1]

In [ ]:
uv3d = (m3d.vertices() + t) @ R.transpose()
mask = np.zeros(m3d.numVertices(), dtype=np.bool)
mask[m3d.boundaryVertices()] = True
uv3d[~mask, 2] *= 0.2
fitter.setVars(uv3d.ravel())

In [ ]:
fitter.setImmersion(m3d.vertices().transpose())

In [ ]:
boundaryVars = np.array([[3 * bvi, 3 * bvi + 1, 3 * bvi + 2] for bvi in m3d.boundaryVertices()]).ravel()

In [ ]:
from tri_mesh_viewer import TriMeshViewer
immersedSurface = mesh.Mesh(fitter.getImmersion().transpose(), F)
import vis
#vf = vis.fields.VectorField(np.pad(uncontractedDirs, [(0, 0), (0, 1)], mode='constant'))
viewer = TriMeshViewer(immersedSurface, width=1024, height=640)
viewer.arrowSize = 40
viewer.showWireframe()
viewer.show()

In [ ]:
fitter.setImmersion(fitter.getImmersion())

In [ ]:
fitter.bendingStiffness = 1e-4
maxDistortion = np.max(np.sqrt(fitter.metricDistSq()))

In [ ]:
fitter.collapsePreventionWeight = 1.0

In [ ]:
fitter.energy(fitter.EnergyType.MetricFitting)

In [ ]:
fitter.energy(fitter.EnergyType.CollapsePrevention)

In [ ]:
import time, matplotlib
from py_newton_optimizer import NewtonOptimizerOptions

opt = NewtonOptimizerOptions()
opt.gradTol = 1e-10
opt.niter = 15

#fixedVars = boundaryVars
fixedVars = fitter.rigidMotionPinVars

metric_fit.benchmark_reset()
for i in range(100):
    cr = metric_fit.fit_metric_newton(fitter, fixedVars, opt)
    if len(cr.energy) < 3: break
    immersedSurface = mesh.Mesh(fitter.getImmersion().transpose(), F)
    sf = vis.fields.ScalarField(np.sqrt(fitter.metricDistSq()), colormap=matplotlib.cm.coolwarm, vmin=0, vmax=maxDistortion)
    viewer.update(mesh=immersedSurface, scalarField=sf)
    time.sleep(0.01)
metric_fit.benchmark_report()